In [ ]:
%pip install duckdb pandas --quiet

In [ ]:
import duckdb

# Connection to DuckDB (in memory)
con = duckdb.connect()

# Disable DuckDB progress bar in notebook outputs
# con.sql("SET enable_progress_bar = false")

In [ ]:
# Convert CSVs to Parquet (skips files that already exist)
import os

os.makedirs("data/parquet", exist_ok=True)

for t in ["movies", "ratings", "links", "tags"]:
    pq = f"data/parquet/{t}.parquet"
    if not os.path.exists(pq):
        con.sql(f"COPY (SELECT * FROM 'data/csv/{t}.csv') TO '{pq}' (FORMAT parquet)")
    # Register Parquet file as a view so queries can use short names
    con.sql(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM '{pq}'")

In [ ]:
# Join Wilhelm Scream DB with MovieLens via IMDb ID (remove 'tt' and cast to integer)
# Excluding 'Invalidated' entries
con.sql("""
    CREATE OR REPLACE TABLE wilhelm AS
    SELECT w.*, l.movieId, m.genres
    FROM 'data/csv/wilhelmscreamdb-2026-11-06.csv' w
    JOIN 'data/csv/links.csv' l
      ON CAST(REPLACE(w."IMDb ID", 'tt', '') AS INTEGER) = l.imdbId
    JOIN 'data/csv/movies.csv' m ON l.movieId = m.movieId
    WHERE w.Verification <> 'Invalidated'
""")

# Count matched movies
con.sql("SELECT COUNT(*) AS n_movies FROM wilhelm").df()

# Baseline - Wilhelm movies vs other movies

In [ ]:
# Baseline - Wilhelm movies vs other movies
# Average rating and popularity (ratings per movie)
con.sql("""
    SELECT
        CASE WHEN movieId IN (SELECT movieId FROM wilhelm) THEN 'Wilhelm' ELSE 'Other' END AS movie_group,
        COUNT(DISTINCT movieId)            AS n_movies,
        ROUND(AVG(rating), 2)              AS avg_rating,
        COUNT(*) / COUNT(DISTINCT movieId) AS ratings_per_movie
    FROM ratings
    GROUP BY movie_group
""").df()

**Baseline – findings**
- Wilhelm movies are only 0.7% of the catalogue (615 of around 84k movies; 2 of the 617 have no ratings).
- They receive approximately 13.6x more ratings per movie (4,715 vs 347) — about 9% of all ratings.
- Average rating is almost identical: 3.60 vs 3.53 (+0.07 on a 0.5–5 scale).

**Conclusion:** 
The Wilhelm Scream appears mostly in highly popular, mainstream movies,
but those movies are not rated noticeably higher than the rest.
*Note: `avg_rating` here is weighted by number of ratings, so popular movies weigh more.*

## Data check
How much information do we really have in the Wilhelm table?

In [ ]:
# List all tables and views in the current DuckDB connection
con.sql("SHOW TABLES").df()

**Data check – findings**
- 617 movies with the Wilhelm Scream matched MovieLens, from 1950 to 2023.
- Scene descriptions are missing for 35% of movies, so Storyline 1 uses the 401 that have one.
- Scream timestamps are missing for 91% of movies, so they are not used.
- Confirmed/disputed votes are almost always 0, so they are not used.

In [ ]:
# Step 1 - Profile every column (min, max, nulls, distinct values, etc.)
con.sql("SUMMARIZE wilhelm").df()

In [ ]:
# Look at 15 example rows (fixed seed so everyone sees the same rows)
con.sql("SELECT * FROM wilhelm USING SAMPLE reservoir(15 ROWS) REPEATABLE (42)").df()

## Storyline 1 – How the extras scream
In what kind of scene is the Wilhelm Scream used? (based on Scene Description)

In [ ]:
# Classify each Wilhelm scene by keyword
con.sql(r"""
CREATE OR REPLACE TABLE scenes AS
SELECT movieId,
       "Scene Description" AS descr,
       CASE
         -- falls / being thrown
         WHEN regexp_matches(lower("Scene Description"), '\b(fall|falls|falling|fell|thrown|throws?|knocked|ledge|cliff|pit|off the|over the|over a|tumbles?|plunges?|jumps?)\b') THEN 'Fall / thrown'
         -- explosions
         WHEN regexp_matches(lower("Scene Description"), 'explo|\b(blast|blasted|bomb|grenade|blown)') THEN 'Explosion'
         -- ranged weapons
         WHEN regexp_matches(lower("Scene Description"), '\b(shot|shoots?|shooting|fires?|firing|gun|guns|gunfire|weapons?|blaster|arrow|laser|rifle)') THEN 'Shot / arrow'
         -- melee combat
         WHEN regexp_matches(lower("Scene Description"), '\b(stab|stabbed|sword|punch|punched|kick|kicked|fight|hit|spear|beat|beats|attacks?)') THEN 'Melee'
         -- creatures / monsters
         WHEN regexp_matches(lower("Scene Description"), '\b(eaten|bitten|monster|creature|dinosaur|zombie|alien)') THEN 'Creature'
         -- vehicles / crashes
         WHEN regexp_matches(lower("Scene Description"), '\b(car|cars|truck|crash|crashes|train|vehicle|plane|drives?|driving|limo|runs over|motorcycle)\b') THEN 'Vehicle'
         ELSE 'Other'
       END AS scene_type
FROM wilhelm
WHERE "Scene Description" IS NOT NULL
""")

# Overall distribution of scene types
con.sql("""
SELECT scene_type,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct  -- share of all scenes
FROM scenes
GROUP BY scene_type
ORDER BY n DESC
""").df()

**Storyline 1 – findings**
- 401 movies with the Wilhelm Scream have a scene description. We sorted each one into a type using keywords.
- Falls are by far the most common type: 134 scenes (33%), more than 3x the next type (fights, with 11%).
- If we only count the 263 scenes where we can tell what is happening, 51% are falls.
- 34% classify as "Other", with mostly vague descriptions that don't say what happens (for example, "At 58 minutes").

**Conclusion:** The Wilhelm Scream is, above all, the sound of someone falling. However, each scene gets only one type (the first keyword found), so the classification is approximate.

## Storyline 2 – You can't escape the scream
What share of MovieLens users rated at least one Wilhelm movie?

In [ ]:
# Share of users who rated at least one Wilhelm movie
con.sql("""
WITH u AS (
    SELECT userId,
           COUNT(*) FILTER (WHERE movieId IN (SELECT movieId FROM wilhelm)) AS n_wilhelm
    FROM ratings
    GROUP BY userId
)
SELECT COUNT(*)                                             AS n_users,
       ROUND(100.0 * COUNT_IF(n_wilhelm > 0) / COUNT(*), 1) AS pct_users_rated_wilhelm,
       MEDIAN(n_wilhelm)                                    AS median_wilhelm_movies_per_user
FROM u
""").df()

**Storyline 2 – findings**
- 95.2% of MovieLens users (200,948 in total) rated at least one movie with the Wilhelm Scream.
- The typical user rated 7 of these movies.
- Why? These movies are only 0.7% of all movies, but they are very popular, so they get about 9% of all ratings.

**Conclusion:** Almost every user has seen at least one movie with the Wilhelm Scream. However, we assume that if a user rated a movie, they watched it, and MovieLens users watch a lot of movies (each one rated at least 20), so they are not a typical audience.

## Storyline 3 – Does anyone notice the scream?
Do MovieLens users mention the Wilhelm Scream in their tags?

In [ ]:
# Is the 'wilhelm' tag spread across many users, or concentrated in a few?
con.sql("""
SELECT userId,
       COUNT(DISTINCT movieId) AS movies_tagged_wilhelm
FROM tags
WHERE lower(tag) LIKE '%wilhelm%'
GROUP BY userId
ORDER BY movies_tagged_wilhelm DESC
""").df()

**Storyline 3 – findings**
- Only 5 users (out of ~200,000) ever tagged a movie with "wilhelm".
- One user tagged 81 movies alone. The other 4 users tagged one movie each.

**Conclusion:** Almost everyone has seen a movie with the Wilhelm Scream (Storyline 2), but almost nobody mentions it. The "wilhelm" tag exists mainly because of one dedicated fan. However, it is worth noting that most users never write tags, so this does not prove that others did not notice the scream.